# Exploratory Data Analysis: Amazon Bestselling Books

This notebook provides interactive EDA for the Amazon bestselling books dataset.

**Author:** Senior Data Science Team  
**Project:** Price Prediction for Bestselling Books

## 1. Setup and Data Loading

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set plot style
sns.set_style('whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Load cleaned data
df = pd.read_csv('../data/cleaned.csv')

print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")

## 2. Data Overview

In [ ]:
# Display first few rows
df.head(10)

In [ ]:
# Data types and missing values
print("Data Types and Missing Values:")
print("="*50)
info_df = pd.DataFrame({
    'Column': df.columns,
    'Type': df.dtypes.values,
    'Non-Null': df.count().values,
    'Null': df.isnull().sum().values,
    'Null %': (df.isnull().sum() / len(df) * 100).values
})
display(info_df)

In [ ]:
# Statistical summary
df.describe()

## 3. Price Analysis

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['price'], bins=30, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].axvline(df['price'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${df["price"].mean():.2f}')
axes[0].axvline(df['price'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: ${df["price"].median():.2f}')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Price Distribution', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(df['price'], vert=True, patch_artist=True)
axes[1].set_ylabel('Price ($)')
axes[1].set_title('Price Box Plot', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Price Statistics:")
print(f"  Mean: ${df['price'].mean():.2f}")
print(f"  Median: ${df['price'].median():.2f}")
print(f"  Std Dev: ${df['price'].std():.2f}")
print(f"  Min: ${df['price'].min():.2f}")
print(f"  Max: ${df['price'].max():.2f}")

## 4. Rating Analysis

In [ ]:
# Rating distribution
plt.figure(figsize=(12, 5))
plt.hist(df['rating'], bins=20, edgecolor='black', alpha=0.7, color='coral')
plt.axvline(df['rating'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["rating"].mean():.2f}')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.title('Rating Distribution', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Rating Statistics:")
print(f"  Mean: {df['rating'].mean():.2f}")
print(f"  Median: {df['rating'].median():.2f}")
print(f"  Std Dev: {df['rating'].std():.2f}")
print(f"  Books with rating >= 4.5: {(df['rating'] >= 4.5).sum()} ({(df['rating'] >= 4.5).sum() / len(df) * 100:.1f}%)")

## 5. Genre Analysis

In [ ]:
# Genre frequency
genre_counts = df['genre'].value_counts().head(10)

plt.figure(figsize=(12, 6))
bars = plt.barh(genre_counts.index, genre_counts.values, color='teal', edgecolor='black', alpha=0.7)
plt.xlabel('Number of Books')
plt.ylabel('Genre')
plt.title('Top 10 Genres by Frequency', fontweight='bold')
plt.gca().invert_yaxis()

for i, bar in enumerate(bars):
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2, f' {int(width)}', ha='left', va='center')

plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Average price by genre
genre_price = df.groupby('genre')['price'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False).head(10)

plt.figure(figsize=(12, 6))
plt.bar(range(len(genre_price)), genre_price['mean'], yerr=genre_price['std'], 
        capsize=5, color='steelblue', edgecolor='black', alpha=0.7)
plt.xticks(range(len(genre_price)), genre_price.index, rotation=45, ha='right')
plt.xlabel('Genre')
plt.ylabel('Average Price ($)')
plt.title('Average Price by Genre (Top 10)', fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

display(genre_price)

## 6. Correlation Analysis

In [ ]:
# Correlation heatmap
numeric_cols = ['price', 'rating', 'reviews_count']
if 'year' in df.columns:
    numeric_cols.append('year')

corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
           square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation with Price:")
print(corr_matrix['price'].sort_values(ascending=False))

## 7. Price vs Features

In [ ]:
# Price vs Reviews
plt.figure(figsize=(12, 6))
plt.scatter(df['reviews_count'], df['price'], alpha=0.5, edgecolors='k', linewidths=0.5, 
           c=df['rating'], cmap='viridis')
plt.colorbar(label='Rating')
plt.xlabel('Reviews Count')
plt.ylabel('Price ($)')
plt.title('Price vs Reviews Count (colored by Rating)', fontweight='bold')
plt.xscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Price vs Rating
plt.figure(figsize=(12, 6))
plt.scatter(df['rating'], df['price'], alpha=0.5, edgecolors='k', linewidths=0.5, color='purple')
plt.xlabel('Rating')
plt.ylabel('Price ($)')
plt.title('Price vs Rating', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Top Authors

In [ ]:
# Top authors by book count
author_counts = df['author'].value_counts().head(10)

plt.figure(figsize=(12, 6))
bars = plt.barh(author_counts.index, author_counts.values, color='orange', edgecolor='black', alpha=0.7)
plt.xlabel('Number of Bestsellers')
plt.ylabel('Author')
plt.title('Top 10 Authors by Bestseller Count', fontweight='bold')
plt.gca().invert_yaxis()

for i, bar in enumerate(bars):
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2, f' {int(width)}', ha='left', va='center')

plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 9. Key Insights

In [ ]:
# Generate key insights
insights = []

insights.append(f"1. Dataset contains {len(df)} books across {df['genre'].nunique()} genres from {df['author'].nunique()} authors.")
insights.append(f"2. Average price is ${df['price'].mean():.2f} with median of ${df['price'].median():.2f}.")
insights.append(f"3. {(df['rating'] >= 4.5).sum()} books ({(df['rating'] >= 4.5).sum() / len(df) * 100:.1f}%) have ratings >= 4.5.")

top_genre = df['genre'].value_counts().index[0]
insights.append(f"4. '{top_genre}' is the most popular genre with {df['genre'].value_counts().iloc[0]} books.")

corr_price_rating = df['price'].corr(df['rating'])
insights.append(f"5. Price and rating correlation: {corr_price_rating:.3f}.")

corr_price_reviews = df['price'].corr(df['reviews_count'])
insights.append(f"6. Price and reviews correlation: {corr_price_reviews:.3f}.")

top_author = df['author'].value_counts().index[0]
insights.append(f"7. '{top_author}' has the most bestsellers with {df['author'].value_counts().iloc[0]} books.")

expensive_threshold = df['price'].quantile(0.9)
insights.append(f"8. Top 10% of books are priced above ${expensive_threshold:.2f}.")

high_reviews = (df['reviews_count'] > 10000).sum()
insights.append(f"9. {high_reviews} books have more than 10,000 reviews.")

insights.append(f"10. Price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}.")

print("\n" + "="*60)
print("TOP 10 KEY INSIGHTS")
print("="*60 + "\n")
for insight in insights:
    print(insight)
print("\n" + "="*60)

## 10. Conclusion

This EDA has revealed key patterns in the Amazon bestselling books dataset:

- **Price Distribution**: Prices vary significantly across genres
- **Rating Patterns**: Most bestsellers have high ratings (>4.0)
- **Genre Insights**: Certain genres dominate the bestseller lists
- **Author Influence**: Some authors have multiple bestsellers
- **Correlations**: Reviews and ratings show varying correlation with price

These insights will inform our price prediction model.